# SHALSTAB Tutorial: Landslide Hazard Modeling in Python

This notebook demonstrates how to use the `shalstab` package for slope stability analysis using the SHALSTAB (Shallow Landsliding STABility) model.

**What you'll learn:**
- How to initialize the SHALSTAB analyzer
- Calculate critical rainfall thresholds
- Analyze stability under different rainfall scenarios
- Compute relative failure probability
- Export results for GIS analysis

**Context:** SHALSTAB is a physically-based model developed by Montgomery & Dietrich (1994) for analyzing rainfall-triggered shallow landslide susceptibility. This implementation is used at [SIATA](https://siata.medellin.gov.co/) — Medellín's early warning system for geohazards.

## 1. Installation

```bash
pip install shalstab
```

In [ ]:
import shalstab
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

print(f"SHALSTAB version: {shalstab.__version__}")
print(f"Training DEM: {shalstab.training_dem}")
print(f"Training geology: {shalstab.training_geology}")

## 2. Initialize the Analyzer

The `Analyzer` class requires:
- `dem_path`: Path to a Digital Elevation Model (GeoTIFF)
- `geo`: Path to geological units (Shapefile/GeoJSON) or a GeoDataFrame
- `geo_columns`: List of 4 column names for geotechnical parameters:
  - Cohesion (kPa)
  - Friction angle (degrees)
  - Unit weight (kN/m³)
  - Permeability (m/s)

In [ ]:
# Initialize with training data
analyzer = shalstab.Analyzer(
    dem_path=shalstab.training_dem,
    geo=shalstab.training_geology,
    geo_columns=["Cohesion", "Phi", "Gamma_kN_m", "Ks_m_s"],
    figsize=(14, 10),
)

print(f"DEM shape: {analyzer.dem.squeeze().shape}")
print(f"DEM bounds: {analyzer.extent}")
print(f"Soil thickness range: {float(analyzer._soil_thickness.min()):.2f} - {float(analyzer._soil_thickness.max()):.2f} m")

## 3. Critical Rainfall Analysis

The critical rainfall is the minimum steady-state precipitation rate required to trigger slope failure at each location. Lower values indicate more susceptible areas.

In [ ]:
# Calculate critical rainfall
critical_rainfall = analyzer.calculate_critical_rainfall(show_plot=True)

# Summary statistics
valid_cr = critical_rainfall.values[~np.isnan(critical_rainfall.values)]
print(f"\nCritical Rainfall Summary (mm/day):")
print(f"  Min:    {valid_cr.min():.1f}")
print(f"  Max:    {valid_cr.max():.1f}")
print(f"  Median: {np.median(valid_cr):.1f}")
print(f"  Mean:   {valid_cr.mean():.1f}")

## 4. Stability Analysis for Multiple Rainfall Scenarios

Classify slope stability under different rainfall intensities. The model assigns each cell to one of four classes:

| Class | Description |
|-------|-------------|
| 1 | Unconditionally Stable |
| 2 | Unconditionally Unstable |
| 3 | Unstable (at given rainfall) |
| 4 | Stable (at given rainfall) |

In [ ]:
# Define rainfall scenarios
scenarios = {
    "Light rain (15 mm/day)": 15,
    "Moderate rain (35 mm/day)": 35,
    "Heavy rain (75 mm/day)": 75,
    "Extreme event (150 mm/day)": 150,
}

results = {}
for name, rainfall in scenarios.items():
    stability, fig = analyzer.calculate_stability(rainfall_mm_day=rainfall)
    plt.close(fig)  # Close to avoid display overload

    # Calculate statistics
    valid = stability.values[~np.isnan(stability.values)]
    total = len(valid)
    unstable_pct = (valid == 3).sum() / total * 100
    stable_pct = (valid == 4).sum() / total * 100
    results[name] = {"unstable": unstable_pct, "stable": stable_pct}

# Print summary table
print(f"{'Scenario':<30} {'Unstable %':>12} {'Stable %':>12}")
print("-" * 56)
for name, stats in results.items():
    print(f"{name:<30} {stats['unstable']:>11.1f}% {stats['stable']:>11.1f}%")

In [ ]:
# Visualize the progression
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
colors_map = {1: 'darkgreen', 2: 'darkred', 3: 'gold', 4: 'lightgreen'}
labels = {1: 'Uncond. Stable', 2: 'Uncond. Unstable', 3: 'Unstable', 4: 'Stable'}

for ax, (name, rainfall) in zip(axes.flat, scenarios.items()):
    stability, _ = analyzer.calculate_stability(rainfall_mm_day=rainfall)
    ax.imshow(analyzer.hillshade, cmap='gray', extent=analyzer.extent)

    # Overlay stability classes
    for class_val, color in colors_map.items():
        mask = stability.values == class_val
        if mask.any():
            masked = np.ma.masked_where(~mask, stability.values)
            ax.imshow(masked, cmap='RdYlGn', alpha=0.5, extent=analyzer.extent, vmin=1, vmax=4)

    ax.set_title(f"{name}", fontsize=12, fontweight='bold')
    ax.set_axis_off()

plt.suptitle('SHALSTAB Stability Analysis \u2014 Rainfall Scenarios', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Failure Probability

The failure probability provides a relative ranking (0–100%) of landslide susceptibility based on the log(q/T) ratio. Higher values indicate areas that require less rainfall to fail.

In [ ]:
# Calculate failure probability
probability = analyzer.calculate_failure_probability()

# Summary
valid_prob = probability.values[~np.isnan(probability.values)]
print(f"\nFailure Probability Summary (%):")
print(f"  Min:    {valid_prob.min():.1f}")
print(f"  Max:    {valid_prob.max():.1f}")
print(f"  Median: {np.median(valid_prob):.1f}")

# Risk zones
high_risk = (valid_prob > 80).sum()
medium_risk = ((valid_prob > 40) & (valid_prob <= 80)).sum()
low_risk = (valid_prob <= 40).sum()
print(f"\nRisk distribution:")
print(f"  High risk (>80%):   {high_risk} cells")
print(f"  Medium risk (40-80%): {medium_risk} cells")
print(f"  Low risk (<40%):    {low_risk} cells")

## 6. Export Results

Export analysis results to GeoTIFF format for use in GIS software (QGIS, ArcGIS, etc.).

In [ ]:
# Export results
import tempfile
from pathlib import Path

output_dir = Path(tempfile.mkdtemp())

# Export critical rainfall
analyzer.export_raster(critical_rainfall, output_dir / "critical_rainfall.tif")

# Export failure probability
analyzer.export_raster(probability, output_dir / "failure_probability.tif")

# Export soil thickness
analyzer.export_raster(analyzer._soil_thickness, output_dir / "soil_thickness.tif")

# Export stability for a specific scenario
stability_50mm, _ = analyzer.calculate_stability(rainfall_mm_day=50)
analyzer.export_raster(stability_50mm, output_dir / "stability_50mm.tif")

print(f"\nFiles exported to: {output_dir}")
for f in output_dir.glob("*.tif"):
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")

## 7. Advanced: Custom Study Area

To use SHALSTAB with your own data, you need:
1. A DEM in projected coordinates (GeoTIFF)
2. Geological units with geotechnical properties (Shapefile or GeoJSON)

```python
# Example with custom data
analyzer = shalstab.Analyzer(
    dem_path="path/to/your_dem.tif",
    geo="path/to/geology.shp",
    geo_columns=["cohesion_kpa", "friction_deg", "gamma_knm3", "k_ms"],
)

# Run the full analysis
critical = analyzer.calculate_critical_rainfall(show_plot=True)
stability, fig = analyzer.calculate_stability(rainfall_mm_day=100)
probability = analyzer.calculate_failure_probability()
```

## References

- Montgomery, D.R., & Dietrich, W.E. (1994). A physically based model for the topographic control on shallow landsliding. *Water Resources Research*, 30(4), 1153–1171.
- Catani, F., Segoni, S., & Falorni, G. (2010). An empirical geomorphology-based approach to the spatial prediction of soil thickness at catchment scale. *Water Resources Research*, 46(5).

---

*Developed at [SIATA](https://siata.medellin.gov.co/) — Medellín's early warning system for geohazards.*